# VDist, CompositeVDist, And Elbo Showcase

This notebook focuses on the lower-level variational inference building blocks. It does not use `LieselVI`; instead, it constructs variational distributions and ELBO objects directly.

In [2]:
import jax
import jax.numpy as jnp
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.optim as opt
import liesel.model as lsl


def target_model(*, seed=1):
    key = jax.random.key(seed)
    y_values = 0.8 + 0.4 * jax.random.normal(key, (24,))
    loc = lsl.Var.new_param(jnp.array(0.0), name="loc")
    shift = lsl.Var.new_param(jnp.array(0.0), name="shift")
    mean = lsl.Calc(lambda loc, shift: loc + shift, loc, shift, _name="mean")
    y = lsl.Var.new_obs(
        y_values,
        lsl.Dist(tfd.Normal, loc=mean, scale=1.0),
        name="y",
    )
    return lsl.Model([y])


model = target_model(seed=1)
{
    "parameters": list(model.parameters),
    "observed": list(model.observed),
}

{'parameters': ['shift', 'loc'], 'observed': ['y']}

## Diagonal Multivariate Normal `VDist`

`VDist` describes one variational block. Calling `.build()` creates the corresponding Liesel variables.

In [3]:
q_diag = opt.VDist(["loc"], model).mvn_diag(scale_diag=0.2).build()

{
    "parameters": q_diag.parameters,
    "var_name": q_diag.var.name,
    "q_parameter_names": list(q_diag.q.parameters),
}

{'parameters': ['(loc)_loc', 'h((loc)_scale)'],
 'var_name': '(loc)',
 'q_parameter_names': ['(loc)_loc', 'h((loc)_scale)']}

## Full-Covariance `VDist`

`mvn_tril` uses a lower-triangular scale parameter. Here the initial scale is supplied explicitly.

In [4]:
q_tril = (
    opt.VDist(list(model.parameters), model)
    .mvn_tril(
        scale_tril=0.1,
        scale_tril_bijector=None,
    )
    .build()
)

{
    "parameters": q_tril.parameters,
    "var_name": q_tril.var.name,
    "position_keys": q_tril.position_keys,
}

{'parameters': ['(loc|shift)_scale_tril', '(loc|shift)_loc'],
 'var_name': '(loc|shift)',
 'position_keys': ['shift', 'loc']}

## Laplace-Style Initialization

The `"laplace"` scale initializer uses the current model position as an approximation to the posterior mode. That assumption should be checked by the user.

In [5]:
laplace_elbo = opt.Elbo.mvn_diag(
    model,
    nsamples=2,
    scale_diag="laplace",
    scale_diag_bijector=None,
)

{
    "vdist_type": type(laplace_elbo.vdist).__name__,
    "q_parameters": laplace_elbo.q.parameters,
    "loss_scaled": laplace_elbo.scale,
}

{'vdist_type': 'VDist',
 'q_parameters': mappingproxy({'(loc|shift)_scale': Var(name="(loc|shift)_scale"),
               '(loc|shift)_loc': Var(name="(loc|shift)_loc")}),
 'loss_scaled': False}

## Composite Variational Distribution

`CompositeVDist` combines simple blocks. This is useful when blocks should be independent or use different families.

In [6]:
q_loc = opt.VDist(["loc"], model).mvn_diag(scale_diag=0.2)
q_shift = opt.VDist(["shift"], model).mvn_diag(scale_diag=0.1)
composite = opt.CompositeVDist(q_loc, q_shift).build()

{
    "parameters": composite.parameters,
    "blocks": [dist.position_keys for dist in composite.vi_dists],
    "q_parameter_count": len(composite.q.parameters),
}

{'parameters': ['(shift)_loc',
  'h((shift)_scale)',
  '(loc)_loc',
  'h((loc)_scale)'],
 'blocks': [['loc'], ['shift']],
 'q_parameter_count': 4}

## ELBO From A Custom `VDist`

`Elbo.from_vdist` turns any built variational distribution into a trainable loss.

In [7]:
split = opt.PositionSplit.from_model(model, share_validate=0.0, shuffle=True, seed=7)
elbo = opt.Elbo.from_vdist(
    composite,
    split,
    nsamples=3,
    scale=True,
)
params = elbo.position(elbo.q.parameters)
value = (
    elbo.evaluate(params, jax.random.key(9), model.state, obs=split.train, nsamples=3)
    / elbo.scalar
)
sample = composite.sample(jax.random.key(10), (4,))

{
    "loss_value": round(float(value), 4),
    "sample_shapes": {key: tuple(value.shape) for key, value in sample.items()},
    "scale_scalar": elbo.scalar,
}

{'loss_value': -1.527,
 'sample_shapes': {'loc': (4,), 'shift': (4,)},
 'scale_scalar': 20}

## Optional Prior Regularization

The `regularize_q_prior` flag controls whether the variational prior contribution is included in the ELBO regularization term.

In [8]:
elbo_no_q_prior = opt.Elbo.from_vdist(
    composite,
    split,
    nsamples=3,
    regularize_q_prior=False,
)

{
    "regularize_q_prior": elbo_no_q_prior.regularize_q_prior,
    "nsamples": elbo_no_q_prior.nsamples,
}

{'regularize_q_prior': False, 'nsamples': 3}